<a href="https://colab.research.google.com/github/tejuuu-7774/GENAI_collabs/blob/main/GenAI_23_03_RAG_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q PyPDF2 sentence-transformers faiss-cpu groq scikit-learn "numpy==2.0.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.7 MB/s eta 0:00:00


In [16]:
import os
import numpy as np
import faiss
import PyPDF2
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder
from google.colab import userdata
GROQ_API_KEY = userdata.get('genai_lab_key')
client = Groq(api_key=GROQ_API_KEY)

In [5]:


print("Loading Bi-Encoder (for fast search) and Cross-Encoder (for precision)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')



Loading Bi-Encoder (for fast search) and Cross-Encoder (for precision)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [9]:

pdf_path = "deep-learning-material-dept-ece-ase-blr-1.pdf"

def extract_text_from_pdf(file_path):
  """Raw extraction of text from the PDF pages."""
  text = ""
  with open(file_path, 'rb') as file:
      reader = PyPDF2.PdfReader(file)
      for page_num in range(len(reader.pages)):
          page_text = reader.pages[page_num].extract_text()
          if page_text:
              text += page_text + "\n"
  return text

print(f"Extracting text from {pdf_path}...")
raw_text = extract_text_from_pdf(pdf_path)



Extracting text from deep-learning-material-dept-ece-ase-blr-1.pdf...


In [10]:

def chunk_text(text, chunk_size=250, overlap=50):

    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk not in chunks:
            chunks.append(chunk)
    return chunks

docs = chunk_text(raw_text)
print(f"Total chunks created: {len(docs)}")


Total chunks created: 590


In [11]:

print("Generating embeddings for all chunks ")
doc_embeddings = embedder.encode(docs)

# Build the FAISS Index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings).astype("float32"))
print(f"Successfully added {index.ntotal} vectors to the FAISS index.")



Generating embeddings for all chunks 
Successfully added 590 vectors to the FAISS index.


In [12]:

def generate_with_groq(prompt):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful teaching assistant. Answer strictly based on the provided textbook context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model="llama-3.1-8b-instant",
        temperature=0.1
    )
    return chat_completion.choices[0].message.content

def ask_pdf(query, initial_k=10, final_k=3):

    # Stage 1: Fast Retrieval via Euclidean distance in FAISS
    query_vector = embedder.encode([query]).astype("float32")
    _, indices = index.search(query_vector, initial_k)
    candidate_docs = [docs[idx] for idx in indices[0]]

    # Stage 2: Reranking for high precision using Cross-Encoder
    pairs = [[query, doc] for doc in candidate_docs]
    scores = reranker.predict(pairs)

    # Sort and pick the absolute best chunks
    ranked_indices = np.argsort(scores)[::-1]
    best_docs = [candidate_docs[i] for i in ranked_indices[:final_k]]

    combined_context = "\n\n---\n\n".join(best_docs)

    # Stage 3: LLM Generation
    prompt = f"""
    Context information from the textbook is below.
    ---------------------
    {combined_context}
    ---------------------
    Given the context information, answer the question: {query}
    """

    return generate_with_groq(prompt)



In [15]:
query = "According to the text, how does the keep gate work in an LSTM unit?"

print(f"\n[User Query]: {query}")
print("-" * 50)
print(f"[Groq Response]:\n{ask_pdf(query)}")


[User Query]: According to the text, how does the keep gate work in an LSTM unit?
--------------------------------------------------
[Groq Response]:
According to the text, the keep gate in an LSTM unit works by determining how much of the previous memory to keep. It does this by computing a bit tensor (a tensor of zeros and ones) that is multiplied with the previous state. If a particular location in the bit tensor holds a 1, it means that location in the memory cell is still relevant and ought to be kept. If that particular location instead holds a 0, it means that the location in the memory cell is no longer relevant and ought to be erased.
